In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.inference.pipeline import RULPipeline
from src.data.loader import load_cmapss_raw
print("Libraries loaded ✓")

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.array(y_true) - np.array(y_pred))**2)))

def mae(y_true, y_pred):
    return float(np.mean(np.abs(np.array(y_true) - np.array(y_pred))))

def nasa_score(y_true, y_pred):
    """Asymmetric penalty: late predictions penalized more than early."""
    d = np.array(y_pred) - np.array(y_true)
    score = np.where(d < 0, np.exp(-d/13) - 1, np.exp(d/10) - 1)
    return float(np.sum(score))

## 1. Load Official Test Set
The official test set contains the LAST cycle of each engine (partial sequence).
Ground truth RUL is provided in RUL_FD001.txt.

In [ ]:
_, test_raw, rul_gt = load_cmapss_raw(data_dir='../data/raw')
y_test = rul_gt.values
print(f"Test engines: {test_raw['engine_id'].nunique()}")
print(f"Ground truth RUL — mean: {y_test.mean():.1f}, range: [{y_test.min()}, {y_test.max()}]")

## 2. LSTM Predictions

In [ ]:
pipeline_lstm = RULPipeline(model_dir='../models', model_type='lstm')
results_lstm  = pipeline_lstm.predict(test_raw, n_mc=50)
results_lstm  = results_lstm.sort_values('engine_id').reset_index(drop=True)
y_pred_lstm   = results_lstm['rul_pred'].values

lstm_rmse  = rmse(y_test,  y_pred_lstm)
lstm_mae   = mae(y_test,   y_pred_lstm)
lstm_nasa  = nasa_score(y_test, y_pred_lstm)
print(f"LSTM  — RMSE: {lstm_rmse:.2f} | MAE: {lstm_mae:.2f} | NASA Score: {lstm_nasa:.0f}")

## 3. XGBoost Predictions

In [ ]:
pipeline_xgb = RULPipeline(model_dir='../models', model_type='xgboost')
results_xgb  = pipeline_xgb.predict(test_raw)
results_xgb  = results_xgb.sort_values('engine_id').reset_index(drop=True)
y_pred_xgb   = results_xgb['rul_pred'].values

xgb_rmse  = rmse(y_test,  y_pred_xgb)
xgb_mae   = mae(y_test,   y_pred_xgb)
xgb_nasa  = nasa_score(y_test, y_pred_xgb)
print(f"XGBoost — RMSE: {xgb_rmse:.2f} | MAE: {xgb_mae:.2f} | NASA Score: {xgb_nasa:.0f}")

## 4. Results Summary

In [ ]:
import os
summary = pd.DataFrame({
    'Model':      ['LSTM', 'XGBoost'],
    'RMSE':       [round(lstm_rmse,  2), round(xgb_rmse,  2)],
    'MAE':        [round(lstm_mae,   2), round(xgb_mae,   2)],
    'NASA Score': [round(lstm_nasa,  0), round(xgb_nasa,  0)],
})
summary['Best RMSE'] = summary['RMSE'] == summary['RMSE'].min()
print(summary.to_string(index=False))

os.makedirs('../docs', exist_ok=True)
summary.to_csv('../docs/metrics_summary.csv', index=False)
print("\nSaved → docs/metrics_summary.csv")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, preds, title in [
    (axes[0], y_pred_lstm, f'LSTM (RMSE={lstm_rmse:.1f})'),
    (axes[1], y_pred_xgb,  f'XGBoost (RMSE={xgb_rmse:.1f})')
]:
    ax.scatter(y_test, preds, alpha=0.5, s=25, color='steelblue')
    lim = [0, max(y_test.max(), preds.max()) + 5]
    ax.plot(lim, lim, 'r--', lw=1.5, label='Perfect prediction')
    ax.set_xlabel('True RUL (cycles)')
    ax.set_ylabel('Predicted RUL (cycles)')
    ax.set_title(title)
    ax.legend(fontsize=9)
plt.suptitle('Predicted vs True RUL — Official Test Set', y=1.01)
plt.tight_layout()
plt.savefig('../docs/prediction_scatter.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Error distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, preds, title in [
    (axes[0], y_pred_lstm, 'LSTM'),
    (axes[1], y_pred_xgb,  'XGBoost')
]:
    errors = np.array(preds) - y_test
    ax.hist(errors, bins=30, edgecolor='black', color='coral', alpha=0.8)
    ax.axvline(0, color='red', linestyle='--', lw=1.5)
    ax.set_xlabel('Prediction Error (pred - true)')
    ax.set_ylabel('Count')
    ax.set_title(f'{title} — Error Distribution')
plt.tight_layout()
plt.savefig('../docs/error_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

## Key Results
Fill in after running:

| Model   | RMSE | MAE | NASA Score |
|---------|------|-----|------------|
| LSTM    |      |     |            |
| XGBoost |      |     |            |

Copy values from the summary table above and paste into README.md.